# HHL Algorithm Walkthrough

This notebook is a guided walkthrough of the repository implementation. It follows the full storytelling arc of the project:

1. Classical linear system
2. Quantum state encoding
3. Phase estimation
4. Eigenvalue inversion
5. Uncomputation
6. Postselection
7. Solution comparison


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import numpy as np
from hhl_lab.analysis import precision_sweep, summarize_spectrum
from hhl_lab.matrices import classical_solution, default_problem, normalized_classical_solution
from hhl_lab.simulation import run_hhl_statevector
from hhl_lab.visualization import (
    plot_eigenvalue_encoding,
    plot_precision_sweep,
    plot_solution_comparison,
    plot_spectral_decomposition,
)

np.set_printoptions(precision=6, suppress=True)


## 1. Classical Problem

The fixed Hermitian system used throughout the repository is

$$
A = \begin{bmatrix} 1 & -1/3 \\ -1/3 & 1 \end{bmatrix},
\qquad
b = \begin{bmatrix} 1 \\ 0 \end{bmatrix},
\qquad
t = 3\pi/4.
$$

The goal of HHL is to prepare a quantum state proportional to the normalized classical solution.

In [ ]:
problem = default_problem()
print('A =')
print(problem.matrix)
print('\nb =')
print(problem.rhs)
print('\nt =', problem.evolution_time)
print('\nClassical solution A^-1 b =')
print(classical_solution(problem))
print('\nNormalized classical state =')
print(normalized_classical_solution(problem))


## 2. Spectral Decomposition

HHL works in the eigenbasis of the Hermitian matrix. The decomposition of $|b\rangle$ tells us how much weight each eigencomponent carries before phase estimation.

In [ ]:
spectrum = summarize_spectrum(problem)
print('Eigenvalues:')
print(spectrum.eigenvalues)
print('\nEigenvectors (columns):')
print(spectrum.eigenvectors)
print('\nEigenbasis coefficients of |b>:')
print(spectrum.eigenbasis_coefficients)


In [ ]:
plot_spectral_decomposition(run_hhl_statevector(problem))


## 3. Why the Chosen Evolution Time Matters

For this example, $t = 3\pi/4$ makes the eigenphases exactly representable on a two-qubit register. That removes approximation error from QPE and makes the circuit especially clean pedagogically.

In [ ]:
result = run_hhl_statevector(problem)
[(step.eigenvalue, step.clock_state, step.rotation_angle) for step in result.artifacts.inversion_steps]


In [ ]:
plot_eigenvalue_encoding(result)


## 4. Full HHL Simulation

Now we run the full circuit, inspect the postselected amplitudes, and compare the recovered state to the classical normalized solution.

In [ ]:
print('Raw postselected amplitudes:')
print(result.raw_solution_amplitudes)
print('\nNormalized HHL solution:')
print(result.normalized_hhl_solution)
print('\nSuccess probability:')
print(result.success_probability)
print('\nState fidelity with normalized classical solution:')
print(result.fidelity_with_classical)
print('\nRelative L2 error:')
print(result.relative_error)


In [ ]:
plot_solution_comparison(result)


The useful HHL branch is the one where the ancilla is measured in $|1\rangle$ and the phase register has been uncomputed back to $|00\rangle$. Those amplitudes are exactly what the package extracts as the recovered solution vector.

In [ ]:
result.artifacts.circuit.draw('text')


## 5. Precision Study

The repository also includes a simple eigenphase-discretization study. The key takeaway is that one phase qubit is insufficient, while two qubits are already exact for this particular example.

In [ ]:
points = precision_sweep(range(1, 7), problem)
[(p.precision_bits, p.phase_bitstrings, p.encoded_eigenvalues, p.fidelity_with_classical, p.l2_error) for p in points]


In [ ]:
plot_precision_sweep(points)


## 6. Interpretation

This project is intentionally small-scale, but it is dense where it matters: the eigendecomposition is explicit, the QPE encoding is exact, the inversion step is inspectable, the postselected branch is extracted directly, and the recovered state is compared numerically to the classical benchmark.